# MNIST Denoising Autoencoder

**Goal:** train a deep learning model that removes noise from handwritten-digit
images using an autoencoder on the MNIST dataset (~60,000 train / ~10,000 test).

The autoencoder receives a **noisy** digit and learns to output the **clean** digit.

**Steps in this notebook**
1. Load and preprocess the MNIST dataset
2. Add artificial noise to create noisy inputs
3. Build and train a denoising autoencoder (noisy → clean)
4. Generate denoised outputs on the test set
5. Visualise results (original / noisy / reconstructed)
6. Analysis, challenges, and observations


## 0. Imports & reproducibility

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


## 1. Load and preprocess the MNIST dataset

We scale pixel values to the `[0, 1]` range (so a `sigmoid` output can reproduce
them) and add a channel dimension so the data fits the convolutional layers:
`(N, 28, 28)` → `(N, 28, 28, 1)`. We don't need the labels — the autoencoder is
trained to reconstruct images, not classify them.


In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

# Normalise to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel axis -> (N, 28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("Train shape:", x_train.shape)
print("Test shape: ", x_test.shape)
print("Pixel range:", x_train.min(), "to", x_train.max())


## 2. Add artificial noise

We corrupt each image with **Gaussian noise** and clip the result back to
`[0, 1]`. `noise_factor` controls how strong the corruption is — `0.5` is a good
balance: clearly noisy, but the digit is still (barely) visible.


In [ ]:
NOISE_FACTOR = 0.5

def add_noise(images, noise_factor=NOISE_FACTOR, seed=None):
    rng = np.random.default_rng(seed)
    noisy = images + noise_factor * rng.normal(0.0, 1.0, size=images.shape)
    return np.clip(noisy, 0.0, 1.0).astype("float32")

x_train_noisy = add_noise(x_train, seed=SEED)
x_test_noisy = add_noise(x_test, seed=SEED + 1)

print("Noisy train shape:", x_train_noisy.shape)


### Quick look: clean vs noisy

In [ ]:
n = 8
plt.figure(figsize=(n * 1.5, 3))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray"); plt.axis("off")
    if i == 0: ax.set_title("Clean", loc="left")
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].squeeze(), cmap="gray"); plt.axis("off")
    if i == 0: ax.set_title("Noisy", loc="left")
plt.tight_layout(); plt.show()


## 3. Build the denoising autoencoder

A symmetric convolutional **encoder–decoder**:

- **Encoder** compresses the 28×28 image down to a 7×7×64 feature map.
- **Decoder** upsamples it back to a full 28×28×1 image.

Convolutions (rather than dense layers) keep spatial structure intact, which
gives much sharper reconstructions. The final `sigmoid` keeps outputs in `[0, 1]`.


In [ ]:
def build_autoencoder():
    inputs = layers.Input(shape=(28, 28, 1), name="noisy_input")

    # Encoder
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D((2, 2), padding="same")(x)        # 28 -> 14
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    encoded = layers.MaxPooling2D((2, 2), padding="same")(x)  # 14 -> 7

    # Decoder
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(encoded)
    x = layers.UpSampling2D((2, 2))(x)                        # 7 -> 14
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    x = layers.UpSampling2D((2, 2))(x)                        # 14 -> 28
    outputs = layers.Conv2D(1, (3, 3), activation="sigmoid",
                            padding="same", name="clean_output")(x)

    model = models.Model(inputs, outputs, name="denoising_autoencoder")
    model.compile(optimizer="adam", loss="binary_crossentropy")
    return model

autoencoder = build_autoencoder()
autoencoder.summary()


## 4. Train the model

Key idea: the **input** is the noisy image and the **target** is the clean image.
The network is forced to learn how to undo the corruption.

> Set `EPOCHS` higher (e.g. 20–50) for better results. Training on CPU is fine
> for MNIST but slower; a GPU makes this a couple of minutes.


In [ ]:
EPOCHS = 20
BATCH_SIZE = 128

history = autoencoder.fit(
    x_train_noisy, x_train,                 # noisy in, clean out
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_test_noisy, x_test),
)


### Training curve

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoch"); plt.ylabel("Binary cross-entropy")
plt.title("Training vs validation loss"); plt.legend(); plt.show()


## 5. Generate denoised outputs on the test set

In [ ]:
denoised = autoencoder.predict(x_test_noisy)
print("Denoised output shape:", denoised.shape)


### Visualise: original / noisy / reconstructed

In [ ]:
n = 10
plt.figure(figsize=(n * 1.5, 4.5))
for i in range(n):
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray"); plt.axis("off")
    if i == 0: ax.set_title("Original", loc="left")

    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].squeeze(), cmap="gray"); plt.axis("off")
    if i == 0: ax.set_title("Noisy", loc="left")

    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(denoised[i].squeeze(), cmap="gray"); plt.axis("off")
    if i == 0: ax.set_title("Denoised", loc="left")
plt.tight_layout(); plt.savefig("results.png", dpi=120, bbox_inches="tight"); plt.show()


## 6. Evaluate quality (PSNR & SSIM)

Beyond eyeballing, we quantify how close the reconstruction is to the clean image:
- **PSNR** (peak signal-to-noise ratio) — higher is better.
- **SSIM** (structural similarity) — closer to 1 is better; captures perceived structure.


In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# Average over the first 1000 test images
N = 1000
psnr_vals, ssim_vals = [], []
for i in range(N):
    clean_i = x_test[i].squeeze()
    den_i = denoised[i].squeeze()
    psnr_vals.append(psnr(clean_i, den_i, data_range=1.0))
    ssim_vals.append(ssim(clean_i, den_i, data_range=1.0))

print(f"Mean PSNR over {N} images: {np.mean(psnr_vals):.2f} dB")
print(f"Mean SSIM over {N} images: {np.mean(ssim_vals):.4f}")


### Save the trained model (used by the Streamlit app)

In [ ]:
autoencoder.save("denoising_autoencoder.keras")
print("Saved denoising_autoencoder.keras")


## 7. Analysis, challenges & observations

**Did it work?**
Yes — the autoencoder removes most of the Gaussian noise while keeping the digit
shape recognisable. Reconstructions are clean and the validation loss tracks the
training loss closely (no major overfitting).

**Observations**
- *Denoising vs sharpness trade-off.* Outputs are slightly blurrier than the
  originals. The decoder reconstructs the most likely clean image, which averages
  over fine details and smooths edges — a known property of reconstruction-loss
  autoencoders.
- *Graceful degradation.* At the trained noise level the output is excellent;
  pushing the noise higher mostly hurts thin digits (1, 7) before thick ones (0, 8).
- *It learns digit structure, not pixels.* Because it only ever sees clean targets,
  the model learns the manifold of valid digit shapes, so it generalises to noise
  patterns it didn't memorise.

**Challenges**
- Choosing a noise factor that is challenging but not destructive (0.5 worked well).
- Avoiding over-smoothing — too deep a bottleneck or too many epochs washes out detail.

**Possible improvements (innovation)**
- Try salt-and-pepper or masking noise instead of Gaussian.
- Add U-Net-style skip connections for sharper edges.
- Compare against a dense (fully-connected) autoencoder baseline.
